# haz: el 33% que no produce NADA, es recuperable con busqueda de ancho fijo?

Auditoria del 21-sep (`docs/AUDITORIA_perdida.md`): el 33% de las tareas no produce **ningun**
candidato y cae al placeholder `[[0]]`. En cinco de esas seis, el modelo esta al **96,3-98,5% por
celda** tras el TTT: la NLL total supera el umbral 4-18 veces **solo por longitud**.

La familia de reglas de aceptacion se cerro el 22-ago, pero **sin TTT y con una sola vista**
(verificado: `dfs_budget.ipynb` no tiene LoRA ni augmentaciones), y el propio plan dejo escrito
"pendiente: re-medir con TTT". Esto es esa re-medicion.

**Instrumento central (UN forward por tarea):** forzar la GT y medir, en cada posicion, el RANGO
del token correcto y su NLL. De ahi sale **analiticamente** el ancho de haz necesario:
`no-argmax = 0` => el greedy ya produce la GT; `ancho ~ exp(deficit frente al camino greedy)`.
Ademas se corre busqueda en haz real de ancho 1 y 8 (coste acotado, sin explosion de ramas,
que fue el modo de fallo del umbral lineal).

TTT de produccion exacto (128 secuencias). 18 tareas de `strat`: 6 con pool vacio y 12 con pool,
estas ultimas como control.


In [ ]:
%%writefile /kaggle/working/_worker_src.py
import glob, hashlib, json, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
T00 = time.time()

SHARD = int(os.environ.get("SHARD", "0"))
N_SHARDS = int(os.environ.get("N_SHARDS", "4"))
N_TAREAS = int(os.environ.get("N_TAREAS", "12"))
N_VISTAS = int(os.environ.get("N_VISTAS", "16"))
BUDGET = float(os.environ.get("BUDGET_VISTA", "45"))
WALL_MAX = float(os.environ.get("WALL_MAX", "2100"))
# UN SOLO BRAZO = produccion exacta. Las tareas las asigna el lanzador (plan LPT por coste).
BRAZO = "r256"
R_LORA = 256
ALPHA = 32
SUCIAS = set(json.loads(os.environ["SUCIAS"]))
OUT = "/kaggle/working/shard%d.json" % SHARD

import numpy as np
import torch
from peft import (LoraConfig, get_peft_model, get_peft_model_state_dict,
                  set_peft_model_state_dict)
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

# --- gate de carga: aborta ESTE shard, no la maquina ---
FICH = "arc-agi_evaluation_challenges.json"
DATA = None
for cand in ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2",
             "/kaggle/input/arc-prize-2026-arc-agi-2"):
    if os.path.exists(os.path.join(cand, FICH)):
        DATA = cand
        break
if DATA is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if FICH in files:
            DATA = root
            break
assert DATA, "sin dataset de la competicion"
CHAL = json.load(open(os.path.join(DATA, FICH)))
SOLS = json.load(open(os.path.join(DATA, "arc-agi_evaluation_solutions.json")))

MDIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and any(x.endswith(".safetensors") for x in files):
        MDIR = root
        break
assert MDIR, "no se encontro el modelo en /kaggle/input"
print("[s%d] modelo en %s" % (SHARD, MDIR), flush=True)

_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
ARC_TOKENS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
_TOKID = torch.tensor(ARC_TOKENS, dtype=torch.long, device="cuda:0")
THR = -float(np.log(0.2))
MAX_SEQ = 3500
LR = 5e-5


def coste(t):
    return sum(len(g["input"]) * len(g["input"][0]) + len(g["output"]) * len(g["output"][0])
               for g in t["train"])


# poblacion ESTRATIFICADA por coste a lo largo de TODA la distribucion: coger las N mas
# baratas fue el modo de fallo de lb33-cap (-2,5 pt). Indices equiespaciados en el ranking.
MIS_KEYS = json.loads(os.environ["MIS_KEYS"])
VACIAS = set(json.loads(os.environ.get("VACIAS", "[]")))
ANCHOS = json.loads(os.environ.get("ANCHOS", "[1, 8]"))
print("[s%d] produccion (r=%d alpha=%d) | %d tareas | costes %s"
      % (SHARD, R_LORA, ALPHA, len(MIS_KEYS), [coste(CHAL[k]) for k in MIS_KEYS]), flush=True)


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens_mask(inp, out):
    t = [IM_START, USER, NL] + grid_tokens(inp) + [EOS]
    m = [0] * len(t)
    head = [IM_START, ASSIST, NL]
    body = grid_tokens(out) + [EOS]
    return t + head + body, m + [0] * len(head) + [1] * len(body)


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


def d4(g, k):
    a = np.array(g)
    if k >= 4:
        a = a.T
    for _ in range(k % 4):
        a = np.rot90(a)
    return a.tolist()


def d4_inv(g, k):
    a = np.array(g)
    for _ in range(k % 4):
        a = np.rot90(a, -1)
    if k >= 4:
        a = a.T
    return a.tolist()


def aplicar(g, k, perm):
    return [[perm[c] for c in fila] for fila in d4(g, k)]


def invertir(g, k, perm):
    inv = [0] * 10
    for i, p in enumerate(perm):
        inv[p] = i
    return d4_inv([[inv[c] for c in fila] for fila in g], k)


def build_vistas(pares, seed, n):
    # vistas de DECODE: produccion usa augment(n=2) = 8 D4 x 2 perms = 16 vistas.
    rng = random.Random(seed)
    vistas = []
    for v in range(n):
        if v == 0:
            k, perm, orden = 0, list(range(10)), list(range(len(pares)))
        else:
            k = rng.randrange(8)
            perm = rng.sample(range(10), 10)
            orden = rng.sample(range(len(pares)), len(pares))
        vistas.append({"k": k, "perm": perm, "orden": orden})
    return vistas


def build_vistas_train(pares, seed, n_perm=16):
    # vistas de TTT: produccion usa augment(n=16) = 8 transformaciones D4 x 16
    # permutaciones de color = 128 SECUENCIAS, cada una vista UNA vez (1 epoca).
    # El PRODUCTO CRUZADO importa: 128 pasos, no 16. La primera version de este banco
    # generaba 16 vistas sueltas -> 8x menos adaptacion -> el 4B no aprendia nada
    # (3 de 16 tareas con GT en alguna vista). El smoke del 2B no lo detecto porque
    # ese modelo memoriza el eval y resuelve sin necesitar TTT.
    rng = random.Random(seed)
    perms = [list(range(10))] + [rng.sample(range(10), 10) for _ in range(n_perm - 1)]
    vistas = []
    for k in range(8):
        for perm in perms:
            vistas.append({"k": k, "perm": perm,
                           "orden": rng.sample(range(len(pares)), len(pares))})
    return vistas


base = AutoModelForCausalLM.from_pretrained(
    MDIR, torch_dtype=torch.bfloat16, device_map="cuda:0", attn_implementation="sdpa")
LCFG = LoraConfig(r=R_LORA, lora_alpha=ALPHA, use_rslora=True, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj",
                                  "embed_tokens", "lm_head"])
MODEL = get_peft_model(base, LCFG)
for _, p_ in MODEL.named_parameters():
    if p_.requires_grad:
        p_.data = p_.data.float()
MODEL.enable_input_require_grads()
MODEL.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
INIT = {k: v.detach().clone() for k, v in get_peft_model_state_dict(MODEL).items()}
try:
    import bitsandbytes as bnb
    OPT_CLS = bnb.optim.AdamW8bit
except Exception:
    OPT_CLS = torch.optim.AdamW
print("[s%d] modelo listo mem=%.1fGB (%.0fs)"
      % (SHARD, torch.cuda.memory_allocated() / 1e9, time.time() - T00), flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


def toks_to_grid(toks):
    rows, row = [], []
    for t in toks:
        if t == NL:
            rows.append(row)
            row = []
        elif 0 <= t <= 9:
            row.append(t)
        else:
            return None
    if row:
        rows.append(row)
    if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
        return None
    return rows


@torch.no_grad()
def dfs(prompt, depth, budget):
    t0 = time.time()
    last, cache0 = prefill(prompt)
    leaves, path = [], []

    def rec(logits, score, cache, left):
        if time.time() - t0 > budget or left <= 1:
            return
        lf = logits.float()
        nll = (score + torch.logsumexp(lf, dim=-1, keepdim=True)
               - lf.index_select(-1, _TOKID)).cpu()
        cands = []
        for j, t in enumerate(ARC_TOKENS):
            sc = nll[0, j].item()
            if sc < THR:
                if t == EOS:
                    leaves.append((sc, list(path)))
                else:
                    cands.append((sc, t))
        for sc, t in sorted(cands):
            if time.time() - t0 > budget:
                return
            with sdpa_kernel(_SDPA):
                o = MODEL(input_ids=torch.tensor([[t]], device="cuda:0", dtype=torch.long),
                          past_key_values=cache, use_cache=True, return_dict=True)
            path.append(t)
            rec(o.logits[:, -1], sc, o.past_key_values, left - 1)
            path.pop()

    rec(last, 0.0, cache0, depth)
    return leaves


def _prompt_canonico(task):
    """Vista canonica (k=0, permutacion identidad), igual que nll_de_gt."""
    p = []
    for par in task["train"]:
        t, _ = pair_tokens_mask(par["input"], par["output"])
        p += t
    return p + query_tokens(task["test"][0]["input"])


@torch.no_grad()
def rangos_de_gt(task, gt):
    """EL INSTRUMENTO CENTRAL, y cuesta UN forward: fuerza la GT y mide, en cada posicion,
    el RANGO del token correcto entre los 12 tokens ARC y su NLL. De ahi sale analiticamente
    que ancho de haz haria falta:
      - rango 1 en todas las posiciones  => el greedy YA produce la GT (ancho 1 basta)
      - deficit acumulado frente al camino greedy => cota del ancho necesario (exp(deficit))"""
    prompt = _prompt_canonico(task)
    gtoks = grid_tokens(gt) + [EOS]
    ids = torch.tensor([prompt + gtoks], device="cuda:0", dtype=torch.long)
    with sdpa_kernel(_SDPA):
        out = MODEL(input_ids=ids, use_cache=False, return_dict=True)
    lg = out.logits[0, len(prompt) - 1:len(prompt) - 1 + len(gtoks)].float()
    lg = lg.index_select(-1, _TOKID)                      # solo los 12 tokens ARC
    lp = lg.log_softmax(-1)
    idx = {t: i for i, t in enumerate(ARC_TOKENS)}
    col = torch.tensor([idx[t] for t in gtoks], device="cuda:0")
    nll_gt = -lp.gather(1, col.view(-1, 1)).squeeze(1)     # NLL del token correcto
    nll_best = -lp.max(dim=1).values                       # NLL del argmax
    rango = (lp > lp.gather(1, col.view(-1, 1))).sum(1) + 1
    deficit = float((nll_gt - nll_best).sum().item())      # cuanto peor es la GT que el greedy
    r = rango.tolist()
    return {"n_tok": len(gtoks), "n_no_argmax": int(sum(1 for x in r if x > 1)),
            "rango_max": int(max(r)), "rangos_altos": sorted([x for x in r if x > 1], reverse=True)[:8],
            "deficit_vs_greedy": round(deficit, 3),
            "ancho_implicado": round(float(np.exp(min(deficit, 20))), 1),
            "nll_total": round(float(nll_gt.sum().item()), 3)}


def _reordena(cache, idx):
    """Reordena/duplica las filas de la cache KV para que coincidan con los padres elegidos.
    Sin esto el haz mezcla la historia de unas secuencias con los tokens de otras (fallo
    silencioso). Se prueban las tres APIs que ha tenido transformers; si ninguna existe se
    LANZA, no se ignora."""
    if hasattr(cache, "batch_select_indices"):
        cache.batch_select_indices(idx)
        return
    if getattr(cache, "key_cache", None) is not None:
        for i in range(len(cache.key_cache)):
            cache.key_cache[i] = cache.key_cache[i].index_select(0, idx)
            cache.value_cache[i] = cache.value_cache[i].index_select(0, idx)
        return
    capas = getattr(cache, "layers", None)
    if capas:
        for c in capas:
            c.keys = c.keys.index_select(0, idx)
            c.values = c.values.index_select(0, idx)
        return
    raise RuntimeError("no se como reordenar esta cache: %s" % type(cache))


@torch.no_grad()
def haz(task, gt, ancho, tope_tok):
    """Busqueda en HAZ de ancho fijo sobre la vista canonica. Coste acotado y predecible:
    siempre `ancho` secuencias vivas, sin explosion de ramas (el modo de fallo del umbral
    lineal). Devuelve si la GT aparece en el haz y el mejor candidato."""
    prompt = _prompt_canonico(task)
    gtoks = grid_tokens(gt) + [EOS]
    ids = torch.tensor([prompt] * ancho, device="cuda:0", dtype=torch.long)
    with sdpa_kernel(_SDPA):
        out = MODEL(input_ids=ids, use_cache=True, return_dict=True)
    cache = out.past_key_values
    lp = out.logits[:, -1].float().index_select(-1, _TOKID).log_softmax(-1)
    vivos = [([], 0.0) for _ in range(ancho)]
    acabados = []
    activo = [True] * ancho
    primero = True
    for paso in range(tope_tok):
        cands = []
        for b in range(ancho):
            if not activo[b]:
                continue
            if primero and b > 0:
                break                                     # todas las filas son iguales al inicio
            cam, sc = vivos[b]
            for j, t in enumerate(ARC_TOKENS):
                cands.append((sc - float(lp[b, j].item()), b, t))
        primero = False
        if not cands:
            break
        cands.sort(key=lambda x: x[0])
        nuevos, filas, toks = [], [], []
        for sc, b, t in cands:
            if len(nuevos) >= ancho:
                break
            cam = vivos[b][0] + [t]
            if t == EOS:
                acabados.append((sc, cam))
                continue
            nuevos.append((cam, sc)); filas.append(b); toks.append(t)
        if not nuevos:
            break
        idxf = torch.tensor(filas, device="cuda:0")
        _reordena(cache, idxf)          # SIN esto las filas de la cache quedan desalineadas
        with sdpa_kernel(_SDPA):
            o = MODEL(input_ids=torch.tensor(toks, device="cuda:0", dtype=torch.long).view(-1, 1),
                      past_key_values=cache, use_cache=True, return_dict=True)
        cache = o.past_key_values
        lp = o.logits[:, -1].float().index_select(-1, _TOKID).log_softmax(-1)
        vivos = nuevos + [([], 1e9)] * (ancho - len(nuevos))
        activo = [True] * len(nuevos) + [False] * (ancho - len(nuevos))
        if len(acabados) >= ancho:
            break
    acabados.sort(key=lambda x: x[0])
    en_haz = any(c == gtoks for _, c in acabados)
    mejor = None
    for _, c in acabados[:1]:
        mejor = toks_to_grid(c[:-1]) if c and c[-1] == EOS else toks_to_grid(c)
    return {"ancho": ancho, "n_acabados": len(acabados), "gt_en_haz": bool(en_haz),
            "mejor_ok": bool(mejor is not None and mejor == gt)}


@torch.no_grad()
def nll_de_gt(task, gt):
    """NLL de la rejilla correcta bajo el modelo YA ADAPTADO, vista canonica (k=0, perm
    identidad). Medida continua: mas potencia que el 0-16 con n=12. Es MECANISMO, no
    veredicto -- no confundir con la loss del TTT, que esta saturada."""
    prompt = []
    for p in task["train"]:
        t, _ = pair_tokens_mask(p["input"], p["output"])
        prompt += t
    prompt += query_tokens(task["test"][0]["input"])
    gtoks = grid_tokens(gt) + [EOS]
    ids = torch.tensor([prompt + gtoks], device="cuda:0", dtype=torch.long)
    with sdpa_kernel(_SDPA):
        out = MODEL(input_ids=ids, use_cache=False, return_dict=True)
    lg = out.logits[0, :-1].float()
    tgt = ids[0, 1:]
    lp = lg.log_softmax(-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    n_p = len(prompt)
    return float(-(lp[n_p - 1:]).sum().item()), len(gtoks)


def entrenar_ttt(task, vistas_train):
    # UNA epoca sobre las vistas: calendario EXACTO de produccion (128 secuencias vistas
    # una vez). Recortarlo es el modo de fallo de ttt05 (26,39) -- no se toca.
    MODEL.config.use_cache = False
    MODEL.train()
    opt = OPT_CLS([p for p in MODEL.parameters() if p.requires_grad], lr=LR)
    seqs = []
    for v in vistas_train:
        toks, mask = [], []
        for idx in v["orden"]:
            p = task["train"][idx]
            t, m = pair_tokens_mask(aplicar(p["input"], v["k"], v["perm"]),
                                    aplicar(p["output"], v["k"], v["perm"]))
            toks += t
            mask += m
        if len(toks) <= MAX_SEQ:
            seqs.append((toks, mask))
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(len(seqs), 1))
    for toks, mask in seqs:
        ids = torch.tensor([toks], device="cuda:0", dtype=torch.long)
        lab = torch.tensor([[t if m else -100 for t, m in zip(toks, mask)]],
                           device="cuda:0", dtype=torch.long)
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=ids, labels=lab)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in MODEL.parameters() if p.requires_grad], 1.0)
        opt.step()
        sch.step()
        opt.zero_grad(set_to_none=True)
    del opt, sch
    MODEL.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    MODEL.eval()
    MODEL.config.use_cache = True
    return len(seqs)


RES = {"tareas": [], "modelo": "4b", "n_vistas": N_VISTAS, "budget_vista": BUDGET,
       "shard": SHARD, "brazo": BRAZO, "r": R_LORA, "alpha": ALPHA}
for i, k in enumerate(MIS_KEYS):
    if time.time() - T00 > WALL_MAX:
        print("[s%d] WALL_MAX alcanzado, corto en %d/%d" % (SHARD, i, len(MIS_KEYS)),
              flush=True)
        break
    task = CHAL[k]
    gt = SOLS[k][0]
    pares = [(p["input"], p["output"]) for p in task["train"]]
    seed = int(hashlib.md5(k.encode()).hexdigest()[:8], 16) % 1024**2

    set_peft_model_state_dict(MODEL, {kk: v.clone() for kk, v in INIT.items()})
    try:
        _pre, _ = nll_de_gt(task, gt)
    except Exception:
        torch.cuda.empty_cache(); _pre = None
    v_train = build_vistas_train(pares, seed)
    assert len(v_train) == 128, "el TTT debe ver 128 secuencias como produccion"
    t0 = time.time()
    n_seqs = entrenar_ttt(task, v_train)
    t_ttt = time.time() - t0

    # === MEDIDAS NUEVAS (sin las 16 vistas: aqui la pregunta es el ANCHO, no la vista) ===
    t0 = time.time()
    try:
        rg = rangos_de_gt(task, gt)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); rg = None
    except Exception as e:
        print("[s%d] rangos fallo en %s: %s" % (SHARD, k, str(e)[:120]), flush=True); rg = None
    t_rang = time.time() - t0
    hazes = []
    tope = min(931, len(grid_tokens(gt)) + 40)
    for anc in ANCHOS:
        t1 = time.time()
        try:
            h = haz(task, gt, anc, tope)
            h["s"] = round(time.time() - t1, 1)
            hazes.append(h)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            hazes.append({"ancho": anc, "oom": True, "s": round(time.time() - t1, 1)})
        except Exception as e:
            print("[s%d] haz %d fallo en %s: %s" % (SHARD, anc, k, str(e)[:120]), flush=True)
            hazes.append({"ancho": anc, "error": str(e)[:120]})
        torch.cuda.empty_cache()
    vistas_out = []
    t_dec = time.time() - t0
    n_con_gt = 0
    try:
        gt_nll, n_gt_tok = nll_de_gt(task, gt)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gt_nll, n_gt_tok = None, None
    except Exception as e:
        print("[s%d] nll_de_gt fallo en %s: %s" % (SHARD, k, str(e)[:120]), flush=True)
        gt_nll, n_gt_tok = None, None
    celdas = len(gt) * len(gt[0])
    RES["tareas"].append({"task": k, "brazo": BRAZO, "r": R_LORA, "vistas": vistas_out,
                          "n_seqs_ttt": n_seqs, "coste": coste(task), "celdas": celdas,
                          "ttt_s": round(t_ttt, 1), "decode_s": round(t_dec, 1),
                          "vistas_con_gt": n_con_gt,
                          "gt_nll": round(gt_nll, 3) if gt_nll is not None else None,
                          "gt_nll_celda": round(gt_nll / celdas, 4) if gt_nll is not None else None,
                          "gt_nll_celda_pre": round(_pre / celdas, 4) if _pre is not None else None,
                          "pool_vacio_en_strat": k in VACIAS,
                          "rangos": rg, "hazes": hazes, "t_rangos": round(t_rang, 1)})
    print("[s%d %d/%d] %s coste %d %s | tok %s no-argmax %s rango_max %s deficit %s ancho~%s | haz %s | ttt %.0fs dec %.0fs"
          % (SHARD, i + 1, len(MIS_KEYS), k, coste(task), "VACIA" if k in VACIAS else "con-pool",
             rg["n_tok"] if rg else "-", rg["n_no_argmax"] if rg else "-",
             rg["rango_max"] if rg else "-", rg["deficit_vs_greedy"] if rg else "-",
             rg["ancho_implicado"] if rg else "-",
             " ".join("w%d:%s" % (h["ancho"], "GT" if h.get("gt_en_haz") else ("oom" if h.get("oom") else "no")) for h in hazes),
             t_ttt, t_dec), flush=True)
    json.dump(RES, open(OUT, "w"))          # persistencia INCREMENTAL por tarea

json.dump(RES, open(OUT, "w"))
print("[s%d] fin %.0fs" % (SHARD, time.time() - T00), flush=True)


In [ ]:
import json, os, subprocess, sys, time
T0 = time.time()
SUCIAS = ["0934a4d8", "136b0064", "16b78196", "981571dc", "aa4ec2a5", "da515329"]
PLAN = [["faa9f03d", "53fb4810", "58f5dbd5", "195c6913"], ["dd6b8c4b", "f931b4a8", "7b3084d4", "89565ca0", "b10624e5"], ["3a25b0d8", "b6f77b65", "4a21e3da", "221dfab4"], ["28a6681f", "21897d95", "d59b0160", "67e490f4", "cb2d8a2c"]]
VACIAS = ["21897d95", "53fb4810", "4a21e3da", "221dfab4", "b10624e5", "195c6913"]

procs = []
for i in range(4):
    env = dict(os.environ)
    env.update({"CUDA_VISIBLE_DEVICES": str(i), "SHARD": str(i), "N_SHARDS": "4",
                "N_VISTAS": "0", "BUDGET_VISTA": "45", "WALL_MAX": "5100",
                "ANCHOS": "[1, 8]", "VACIAS": json.dumps(VACIAS),
                "SUCIAS": json.dumps(SUCIAS), "MIS_KEYS": json.dumps(PLAN[i])})
    fh = open("/kaggle/working/log%d.txt" % i, "w")
    procs.append((i, subprocess.Popen([sys.executable, "/kaggle/working/_worker_src.py"],
                                      env=env, stdout=fh, stderr=subprocess.STDOUT), fh))
    print("worker %d: %d tareas" % (i, len(PLAN[i])), flush=True)
for i, p, fh in procs:
    p.wait(); fh.close()
    print("worker %d rc=%d a los %.0fs" % (i, p.returncode, time.time() - T0), flush=True)

tareas = []
for i in range(4):
    f = "/kaggle/working/shard%d.json" % i
    if os.path.exists(f):
        tareas += json.load(open(f)).get("tareas", [])
json.dump({"tareas": tareas}, open("/kaggle/working/haz.json", "w"))
for i in range(4):
    print("=" * 12 + (" LOG WORKER %d " % i) + "=" * 12, flush=True)
    try:
        print(open("/kaggle/working/log%d.txt" % i).read()[-2500:], flush=True)
    except Exception as e:
        print("sin log:", e, flush=True)

tareas.sort(key=lambda t: (not t.get("pool_vacio_en_strat"), t["coste"]))
print("")
print("=" * 104, flush=True)
print("HAZ: en las tareas donde el DFS no produce NADA, que ancho haria falta para tener la GT?", flush=True)
print("=" * 104, flush=True)
print("%-10s %9s %6s %10s %9s %9s %10s | %s" % ("tarea", "pool strat", "tok", "no-argmax", "rango max", "deficit", "ancho~", "haz 1/8"), flush=True)
for t in tareas:
    r = t.get("rangos") or {}
    hs = {h["ancho"]: h for h in (t.get("hazes") or [])}
    def cel(a):
        h = hs.get(a)
        if not h: return "-"
        if h.get("oom"): return "oom"
        return "GT" if h.get("gt_en_haz") else "no"
    print("%-10s %9s %6s %10s %9s %9s %10s | %s" % (
        t["task"], "VACIO" if t.get("pool_vacio_en_strat") else "con-pool",
        r.get("n_tok", "-"), r.get("n_no_argmax", "-"), r.get("rango_max", "-"),
        r.get("deficit_vs_greedy", "-"), r.get("ancho_implicado", "-"),
        "  ".join("%s:%s" % (a, cel(a)) for a in (1, 8))), flush=True)

vac = [t for t in tareas if t.get("pool_vacio_en_strat") and t.get("rangos")]
con = [t for t in tareas if not t.get("pool_vacio_en_strat") and t.get("rangos")]
def med(xs):
    s = sorted(xs); n = len(s)
    return s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2.0
print("")
for nom, g in (("POOL VACIO", vac), ("con pool", con)):
    if not g: continue
    print("%-11s n=%d | no-argmax mediana %.0f de %.0f tokens (%.2f%%) | deficit mediana %.2f | ancho implicado mediana %.0f"
          % (nom, len(g), med([t["rangos"]["n_no_argmax"] for t in g]), med([t["rangos"]["n_tok"] for t in g]),
             100.0 * med([t["rangos"]["n_no_argmax"] for t in g]) / max(med([t["rangos"]["n_tok"] for t in g]), 1),
             med([t["rangos"]["deficit_vs_greedy"] for t in g]),
             med([t["rangos"]["ancho_implicado"] for t in g])), flush=True)
    for a in (1, 8):
        n = sum(1 for t in g if any(h.get("ancho") == a and h.get("gt_en_haz") for h in (t.get("hazes") or [])))
        print("      ancho %2d: GT en el haz en %d/%d" % (a, n, len(g)), flush=True)
print("")
print("LECTURA: no-argmax = 0 => el GREEDY ya produce la GT y basta ancho 1.", flush=True)
print("         deficit = cuanto peor puntua la GT que el camino greedy; ancho~ = exp(deficit)", flush=True)
print("         es la cota del ancho necesario para que la GT sobreviva en el haz.", flush=True)
print("RESULT_JSON:" + json.dumps({"tareas": len(tareas), "wall_s": round(time.time() - T0)}), flush=True)
